# T1, T2, and Ramsey experiments (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook collects the three standard coherence measurements:
energy relaxation (T1), echo-based dephasing (T2), and Ramsey detuning.

## 1. Create an `Experiment`

Pick at least one qubit from the device configuration.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from IPython.display import display

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)


def display_experiment_result(result):
    """Display per-target fit-style figures from an ExperimentResult."""
    for data in result.data.values():
        display(data.plot(return_figure=True))


## 2. Connect to the setup

Connect before you run any checks or coherence measurements.

In [2]:
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

## 3. Check the waveform

Use a waveform check to confirm that the readout path is behaving normally.

In [3]:
waveform_result = exp.check_waveform()

## 4. Prepare the half-pi pulse

A baseline Rabi fit and half-pi calibration make the later coherence measurements easier to interpret.

In [4]:
rabi_result = exp.obtain_rabi_params()
hpi_result = exp.calibrate_hpi_pulse()
exp.calib_note.save()

## 5. Run the T1 experiment

Sweep a long wait range on a logarithmic grid to extract the energy-relaxation time.

Here `np.geomspace(100, 200e3, 31)` means a delay sweep from 100 ns to 200 us with logarithmic spacing. This covers both the short-time region where the signal still starts near its initial value and the long-time tail where the excited-state population has mostly decayed, so T1 can be estimated without oversampling only the slow end.

In [5]:
t1_result = exp.t1_experiment(
    exp.qubit_labels,
    time_range=np.geomspace(100, 200e3, 31),
    plot=False,
    save_image=True,
)

display_experiment_result(t1_result)


Target qubits: ['Q00', 'Q01', 'Q02', 'Q03']
Subgroups: [['Q00', 'Q01', 'Q02', 'Q03']]
(1/1) Conducting T1 experiment for ['Q00', 'Q01', 'Q02', 'Q03']...



## 6. Run the T2 echo experiment

Use an echo-style sequence over a logarithmic time range to estimate the dephasing time.

The same `np.geomspace(100, 200e3, 31)` range corresponds to total echo delays from 100 ns to 200 us. Physically, this lets you see both the early coherent regime and the later decay envelope in one sweep, which is useful when the expected T2 is not known in advance.

In [6]:
t2_result = exp.t2_experiment(
    exp.qubit_labels,
    time_range=np.geomspace(100, 200e3, 31),
    plot=False,
    save_image=True,
)

display_experiment_result(t2_result)


Target qubits: ['Q00', 'Q01', 'Q02', 'Q03']
Subgroups: [['Q00', 'Q01', 'Q02', 'Q03']]
(1/1) Conducting T2 experiment for ['Q00', 'Q01', 'Q02', 'Q03']...



## 7. Run the Ramsey experiment

Use a small detuning to measure the Ramsey fringes and estimate the detuning-sensitive coherence.

In [7]:
ramsey_result = exp.ramsey_experiment(
    exp.qubit_labels,
    time_range=np.arange(0, 10_001, 100),
    detuning=0.001,  # 0.001 GHz = 1 MHz detuning
    plot=False,
)

display_experiment_result(ramsey_result)


Target qubits: ['Q00', 'Q01', 'Q02', 'Q03']
Spectator qubits: []
Bare frequency with |0>:
  Q00: 7.157231

Bare frequency with |0>:
  Q01: 8.032295

Bare frequency with |0>:
  Q02: 7.812112

Bare frequency with |0>:
  Q03: 6.944337



## 8. Reload the updated control frequency and verify it

After editing the frequency file, reload and confirm the effect with one more Rabi measurement.

In [8]:
# Update `control_frequency.yaml` manually, then reload
exp.reload()

# Re-check the Rabi oscillation
updated_rabi_result = exp.obtain_rabi_params()